# AWA2 — CREAM Training Analysis
Loads TensorBoard event files and intervention results from the AWA2 CREAM experiment.
- Section 1: Loss & accuracy curves from TensorBoard
- Section 2: Intervention curves

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

EXPERIMENTS_ROOT = Path('/home/dani00003/mCREAM/experiments')

# Path to the AWA2 CREAM lightning_logs version folder
AWA2_CREAM_DIR = EXPERIMENTS_ROOT / 'AWA2' / 'train_cbm' / 'Standard_AWA2_resnet101'

print('CREAM dir exists:', AWA2_CREAM_DIR.exists())
for p in sorted(AWA2_CREAM_DIR.rglob('events.out.*')):
    print(' ', p.relative_to(EXPERIMENTS_ROOT))

---
# Section 1 — Loss & Accuracy Curves

In [ ]:
def load_tensorboard(log_dir):
    """Load all scalar tags from a TensorBoard event file directory."""
    ea = EventAccumulator(str(log_dir))
    ea.Reload()
    tags = ea.Tags()['scalars']
    data = {}
    for tag in tags:
        events = ea.Scalars(tag)
        data[tag] = pd.DataFrame({
            'step':  [e.step  for e in events],
            'epoch': [e.step  for e in events],  # reuse step as proxy
            'value': [e.value for e in events],
        })
    return data


# Find the latest version folder
version_dirs = sorted(AWA2_CREAM_DIR.rglob('lightning_logs/version_*'))
if not version_dirs:
    print('No version dirs found yet — job may still be running.')
else:
    TB_DIR = version_dirs[-1]
    print('Loading from:', TB_DIR)
    tb = load_tensorboard(TB_DIR)
    print('Available tags:', list(tb.keys()))

In [ ]:
# Tags to plot — adjust if your run uses different names
LOSS_TAGS = [t for t in tb if 'loss' in t.lower()]
ACC_TAGS  = [t for t in tb if 'acc' in t.lower() or 'accuracy' in t.lower()]
CONCEPT_TAGS = [t for t in tb if 'concept' in t.lower()]

def plot_tags(tag_list, title, ylabel, ncols=2):
    if not tag_list:
        print(f'No tags for: {title}')
        return
    nrows = int(np.ceil(len(tag_list) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 4 * nrows), squeeze=False)
    fig.suptitle(title, fontsize=12, fontweight='bold')
    for i, tag in enumerate(tag_list):
        ax = axes[i // ncols][i % ncols]
        df = tb[tag]
        ax.plot(df['step'], df['value'], lw=2)
        ax.set_title(tag, fontsize=9)
        ax.set_xlabel('Step')
        ax.set_ylabel(ylabel)
    # hide unused axes
    for j in range(i + 1, nrows * ncols):
        axes[j // ncols][j % ncols].set_visible(False)
    plt.tight_layout()
    plt.show()

plot_tags(LOSS_TAGS,    'AWA2 CREAM — Loss curves',            'Loss')
plot_tags(ACC_TAGS,     'AWA2 CREAM — Accuracy curves',        'Accuracy')
plot_tags(CONCEPT_TAGS, 'AWA2 CREAM — Concept metric curves',  'Value')

In [ ]:
# Overlay train vs val for loss and task accuracy
pairs = [
    ('train_loss', 'val_loss',         'Loss (train vs val)'),
    ('train_task_accuracy', 'val_task_accuracy', 'Task Accuracy (train vs val)'),
    ('train_concept_accuracy', 'val_concept_accuracy', 'Concept Accuracy (train vs val)'),
]

for train_tag, val_tag, title in pairs:
    # fuzzy match — handle step_* prefixes lightning adds
    t_key = next((k for k in tb if train_tag in k), None)
    v_key = next((k for k in tb if val_tag   in k), None)
    if t_key is None and v_key is None:
        print(f'Skipping {title}: tags not found')
        continue
    fig, ax = plt.subplots(figsize=(8, 4))
    if t_key:
        df = tb[t_key]
        ax.plot(df['step'], df['value'], label='train', lw=2)
    if v_key:
        df = tb[v_key]
        ax.plot(df['step'], df['value'], label='val', lw=2, ls='--')
    ax.set_title(f'AWA2 CREAM — {title}', fontweight='bold')
    ax.set_xlabel('Step'); ax.legend()
    plt.tight_layout()
    plt.show()

---
# Section 2 — Intervention Curves

In [ ]:
# Find intervention_results.csv
iv_files = sorted(AWA2_CREAM_DIR.rglob('intervention_results.csv'))
if not iv_files:
    print('No intervention_results.csv found yet.')
else:
    iv = pd.concat([pd.read_csv(f) for f in iv_files], ignore_index=True)
    print('Columns:', list(iv.columns))
    print(iv.head())

In [ ]:
if 'iv' in dir() and len(iv) > 0:
    has_group = 'group_interventions' in iv.columns

    fig, axes = plt.subplots(1, 2 if has_group else 1,
                             figsize=(14 if has_group else 7, 5),
                             sharey=True)
    if not has_group:
        axes = [axes]
    fig.suptitle('AWA2 CREAM — Intervention curves', fontsize=12, fontweight='bold')

    for ax, group_flag, label in [
        (axes[0], False, 'Individual concept interventions'),
        *( [(axes[1], True,  'Group interventions')] if has_group else [] )
    ]:
        subset = iv[iv['group_interventions'] == group_flag] if has_group else iv
        agg = subset.groupby('num_interventions')['test_task_accuracy'].agg(['mean', 'std']).reset_index()
        ax.plot(agg['num_interventions'], agg['mean'], lw=2.5, marker='o', markersize=5, color='#2980b9')
        ax.fill_between(agg['num_interventions'],
                        agg['mean'] - agg['std'],
                        agg['mean'] + agg['std'],
                        alpha=0.2, color='#2980b9')
        ax.set_title(label, fontsize=10)
        ax.set_xlabel('Number of concepts intervened on')
        ax.set_ylabel('Task Accuracy')
        ax.set_ylim(bottom=0)

    plt.tight_layout()
    plt.show()

    # Print final accuracy at full intervention
    max_iv = agg.iloc[-1]
    print(f"Full intervention ({int(max_iv['num_interventions'])} concepts): "
          f"{max_iv['mean']:.4f} ± {max_iv['std']:.4f}")